# Data Exploration: MNIST PU Dataset

This notebook explores the MNIST Positive-Unlabeled (PU) dataset used in the nnPU paper reproduction.

**Dataset Configuration:**
- Positive class: Even digits (0, 2, 4, 6, 8)
- Negative class: Odd digits (1, 3, 5, 7, 9)
- Labeled positive samples: 100
- Unlabeled samples: ~59,900 (contains both positive and negative)
- Class prior π: ~0.49

In [ ]:
from collections import defaultdict

import holoviews as hv
import hvplot.pandas  # noqa: F401
import pandas as pd
from bokeh.layouts import gridplot
from bokeh.models import Title
from bokeh.plotting import figure, show
from torch.utils.data import DataLoader

from pu_learning.data import MNISTPUDataset
from pu_learning.utils.reproducibility import set_seed

# Enable Bokeh backend
hv.extension("bokeh")

# Set reproducible seed
set_seed(42)

## 1. Load Dataset

In [ ]:
# Create dataset with default configuration
train_dataset = MNISTPUDataset(
    root="../data",
    n_positive=100,
    positive_class="even",
    train=True,
    download=True,
)

test_dataset = MNISTPUDataset(
    root="../data",
    n_positive=100,  # Not used for test set, but required parameter
    positive_class="even",
    train=False,
    download=True,
)

print(f"Training set size: {len(train_dataset):,}")
print(f"Test set size: {len(test_dataset):,}")
print(f"Class prior π: {train_dataset.class_prior:.4f}")

## 2. Dataset Statistics

In [ ]:
# Count labeled positive, unlabeled positive, unlabeled negative
n_labeled_pos = train_dataset.is_labeled.sum().item()
n_unlabeled = (~train_dataset.is_labeled).sum().item()

# Get unlabeled indices
unlabeled_mask = ~train_dataset.is_labeled
n_unlabeled_pos = (train_dataset.targets[unlabeled_mask] == 1).sum().item()
n_unlabeled_neg = (train_dataset.targets[unlabeled_mask] == 0).sum().item()

print("Training Set Breakdown:")
print(f"  Labeled Positive (P): {n_labeled_pos:,}")
print(f"  Unlabeled (U): {n_unlabeled:,}")
print(f"    - True Positive: {n_unlabeled_pos:,}")
print(f"    - True Negative: {n_unlabeled_neg:,}")
print(f"\nTotal True Positive: {n_labeled_pos + n_unlabeled_pos:,}")
print(f"Total True Negative: {n_unlabeled_neg:,}")
print(f"Estimated class prior: {(n_labeled_pos + n_unlabeled_pos) / len(train_dataset):.4f}")

## 3. Visualize Sample Images

Interactive visualization using Bokeh. Hover over images to see details.

In [ ]:
# Get indices for different sample types
labeled_pos_indices = train_dataset.get_positive_labeled_indices()
unlabeled_indices = train_dataset.get_unlabeled_indices()

# Get unlabeled samples split by true label
unlabeled_pos_indices = [
    idx.item()
    for idx in unlabeled_indices
    if train_dataset.targets[idx] == 1
][:5]
unlabeled_neg_indices = [
    idx.item()
    for idx in unlabeled_indices
    if train_dataset.targets[idx] == 0
][:5]


def create_image_figure(img_tensor, title_text):
    """Create a Bokeh figure for displaying an MNIST image."""
    # img_tensor is already normalized, convert back to 0-255 for display
    img_denorm = img_tensor * train_dataset.MNIST_STD + train_dataset.MNIST_MEAN
    img_array = img_denorm.reshape(28, 28).numpy()
    p = figure(
        width=120,
        height=120,
        toolbar_location=None,
        x_range=(0, 28),
        y_range=(28, 0),
    )
    p.image(image=[img_array], x=0, y=0, dw=28, dh=28, palette="Greys256")
    p.axis.visible = False
    p.grid.visible = False
    p.add_layout(Title(text=title_text, text_font_size="10pt"), "above")
    return p


# Create grid of images
row1 = []  # Labeled Positive
row2 = []  # Unlabeled (True Positive)
row3 = []  # Unlabeled (True Negative)

for idx in range(5):
    # Row 1: Labeled Positive
    img, target, is_labeled = train_dataset[labeled_pos_indices[idx]]
    digit = train_dataset.original_labels[labeled_pos_indices[idx]].item()
    row1.append(create_image_figure(img, f"Labeled P\nDigit: {digit}"))

    # Row 2: Unlabeled (True Positive)
    img, target, is_labeled = train_dataset[unlabeled_pos_indices[idx]]
    digit = train_dataset.original_labels[unlabeled_pos_indices[idx]].item()
    row2.append(create_image_figure(img, f"Unlabeled\nDigit: {digit} (Even)"))

    # Row 3: Unlabeled (True Negative)
    img, target, is_labeled = train_dataset[unlabeled_neg_indices[idx]]
    digit = train_dataset.original_labels[unlabeled_neg_indices[idx]].item()
    row3.append(create_image_figure(img, f"Unlabeled\nDigit: {digit} (Odd)"))

# Create grid layout
grid = gridplot([row1, row2, row3], toolbar_location=None)
show(grid)

## 4. Distribution of Digits

Interactive bar charts showing digit distribution across dataset categories.

In [ ]:
# Count digits in each category
digit_counts = {
    "labeled_pos": defaultdict(int),
    "unlabeled_pos": defaultdict(int),
    "unlabeled_neg": defaultdict(int),
}

# Get labeled positive indices
labeled_indices = train_dataset.get_positive_labeled_indices()
for idx in labeled_indices:
    digit = train_dataset.original_labels[idx].item()
    digit_counts["labeled_pos"][digit] += 1

# Get unlabeled indices
unlabeled_indices = train_dataset.get_unlabeled_indices()
for idx in unlabeled_indices:
    digit = train_dataset.original_labels[idx].item()
    if train_dataset.targets[idx] == 1:
        digit_counts["unlabeled_pos"][digit] += 1
    else:
        digit_counts["unlabeled_neg"][digit] += 1


# Create DataFrames for plotting
def create_digit_df(counts_dict, category_name):
    df = pd.DataFrame([
        {"Digit": digit, "Count": count, "Category": category_name}
        for digit, count in sorted(counts_dict.items())
    ])
    # Fill in missing digits with 0
    all_digits = pd.DataFrame({"Digit": range(10)})
    df = all_digits.merge(df, on="Digit", how="left").fillna(
        {"Count": 0, "Category": category_name}
    )
    return df


df_labeled = create_digit_df(digit_counts["labeled_pos"], "Labeled Positive (100)")
df_unlabeled_pos = create_digit_df(
    digit_counts["unlabeled_pos"], "Unlabeled (True Positive)"
)
df_unlabeled_neg = create_digit_df(
    digit_counts["unlabeled_neg"], "Unlabeled (True Negative)"
)

# Create interactive plots
plot1 = df_labeled.hvplot.bar(
    x="Digit",
    y="Count",
    title="Labeled Positive (100)",
    color="#1f77b4",
    width=350,
    height=300,
    ylim=(0, None),
)

plot2 = df_unlabeled_pos.hvplot.bar(
    x="Digit",
    y="Count",
    title="Unlabeled (True Positive)",
    color="#2ca02c",
    width=350,
    height=300,
    ylim=(0, None),
)

plot3 = df_unlabeled_neg.hvplot.bar(
    x="Digit",
    y="Count",
    title="Unlabeled (True Negative)",
    color="#ff7f0e",
    width=350,
    height=300,
    ylim=(0, None),
)

# Display side by side
(plot1 + plot2 + plot3).cols(3)

## 5. Test Set Distribution

In [ ]:
# Count even vs odd in test set
n_test_pos = (test_dataset.targets == 1).sum().item()
n_test_neg = (test_dataset.targets == 0).sum().item()

print("Test Set Breakdown:")
print(f"  Even digits (Positive): {n_test_pos:,}")
print(f"  Odd digits (Negative): {n_test_neg:,}")
print(f"  Class balance: {n_test_pos / len(test_dataset):.4f}")

# Plot test set distribution by digit
test_digit_counts = defaultdict(int)
for idx in range(len(test_dataset)):
    digit = test_dataset.original_labels[idx].item()
    test_digit_counts[digit] += 1

# Create DataFrame
test_df = pd.DataFrame([
    {
        "Digit": digit,
        "Count": count,
        "Class": "Even (Positive)" if digit % 2 == 0 else "Odd (Negative)",
    }
    for digit, count in sorted(test_digit_counts.items())
])

# Interactive plot with color-coded bars
test_df.hvplot.bar(
    x="Digit",
    y="Count",
    by="Class",
    title="Test Set Digit Distribution",
    width=800,
    height=400,
    legend="top_right",
    color=["#1f77b4", "#ff7f0e"],
)

## 6. Data Loader Example

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# Get a batch
images, targets, is_labeled = next(iter(train_loader))

print("\nBatch shapes:")
print(f"  Images: {images.shape}")
print(f"  Targets: {targets.shape}")
print(f"  Is labeled: {is_labeled.shape}")
print("\nBatch statistics:")
print(f"  Labeled positive in batch: {is_labeled.sum().item()}")
print(f"  Unlabeled in batch: {(~is_labeled).sum().item()}")
print(f"  True positive in batch: {targets.sum().item()}")

## Summary

This notebook explored the MNIST PU dataset configuration used for reproducing the nnPU paper:

1. **Training set**: 100 labeled positive + ~59,900 unlabeled samples
2. **Test set**: ~10,000 fully labeled samples
3. **Positive class**: Even digits (0, 2, 4, 6, 8)
4. **Negative class**: Odd digits (1, 3, 5, 7, 9)
5. **Class prior π**: ~0.49 (approximately balanced)

All visualizations are interactive - you can hover, pan, and zoom to explore the data!

The dataset is ready for training PN, uPU, and nnPU models!